# 08b — Tracking and teams on in-domain DFL footage

**In-domain rerun on DFL clip 121364_9** (Bundesliga Data Shootout, via HF `dbal0503/Bundesliga`, pinned revision). Same broadcast camera as the training data; match 121364 is in the detection **and** pitch-keypoint **test** splits, so no model saw it. The original competition licence could not be verified: report metrics and stills with credit to DFL Deutsche Fussball Liga, and **do not publish the rendered video**.

# 08 — Tracking and team assignment (Phase 2, demo)

**Clip:** "Match de football France-Allemagne - 16 octobre 2018 - Phase de jeu (1)" by Like tears in rain, Wikimedia Commons, **CC BY-SA 4.0** — https://commons.wikimedia.org/wiki/File:Match_de_football_France-Allemagne_-_16_octobre_2018_-_Phase_de_jeu_(1).ogv. 1920×1080, 30 fps, 17 s. The annotated output is a derivative and is shared under CC BY-SA 4.0 with this attribution.

**Domain shift, stated up front:** the clip is filmed from the stands at a low angle, not from the broadcast camera the detector was trained on. Detection and tracking numbers here describe the demo clip, not the test split.

- **Tracking:** `model.track(..., tracker="bytetrack.yaml", persist=True)` — tracker named explicitly (ultralytics 8.4.163 defaults to `tracktrack.yaml`).
- **ID switches:** the clip has no ground-truth identities, so true ID switches cannot be counted. Reported instead as fragmentation proxies: player track IDs per minute and median track length.
- **Ball track continuity:** share of frames with a live ball track. Run for each detector in `WEIGHTS` to link it back to ball mAP from the study.
- **Fragmentation diagnostic:** the clip is handheld phone video. If camera shake breaks IoU association, BoT-SORT (camera-motion compensation) should fragment far less than ByteTrack on the same detections. The rendered video stays ByteTrack as specified; BoT-SORT is only the check.
- **Teams:** torso crop, pitch-green masked, mean Lab colour, KMeans k=2 over all player crops in the clip, referees and goalkeepers excluded by class, one team per track by majority vote. `agreement` = how consistently a track's frames agree with its vote.
- **Output:** annotated MP4 — players coloured by team, ball with a 30-frame fading trail (gaps in the trail are detection dropouts).

In [ ]:
VIDEO = "/content/video/dfl_121364_9.mp4"   # H.264 copy: ultralytics does not read .ogv
VIDEO_URL = "https://huggingface.co/datasets/dbal0503/Bundesliga/resolve/8f854e3e4f7007134410f2040827bba7bf4c3dd8/Bundesliga/Clips/121364_9.mp4"
WEIGHTS = {  # label -> (weights, imgsz it was trained at)
    "02 baseline (640)": ("/content/results/baseline_yolo11s_640/best.pt", 640),
    "03 imgsz 1280": ("/content/results/res_yolo11s_1280/best.pt", 1280),
}
RENDER_WITH = "03 imgsz 1280"  # best detector of the study draws the output video
OUT = "/content/results/tracking_dfl"
TRACKERS = ["bytetrack.yaml", "botsort.yaml"]  # botsort adds camera-motion compensation (GMC): diagnostic only

In [ ]:
import sys, json, pathlib, subprocess, urllib.request
sys.path.insert(0, "/content/ballhawk")
import pandas as pd
from ultralytics import YOLO
import ballhawk_video as bv

out = pathlib.Path(OUT); out.mkdir(parents=True, exist_ok=True)
VIDEO = bv.fetch_clip(VIDEO, VIDEO_URL)
fps = bv.video_fps(VIDEO)

In [ ]:
stats, tracks = {}, {}
for label, (w, imgsz) in WEIGHTS.items():
    names = [YOLO(w).names[i] for i in range(len(YOLO(w).names))]
    for tracker in TRACKERS:
        key = (label, tracker)
        tracks[key] = bv.track(w, VIDEO, tracker=tracker, imgsz=imgsz)
        stats[key] = bv.track_stats(tracks[key], fps, names.index("player"), names.index("ball"))
stats_df = pd.DataFrame(stats).T.round(3)
stats_df

In [ ]:
names = [YOLO(WEIGHTS[RENDER_WITH][0]).names[i] for i in range(len(YOLO(WEIGHTS[RENDER_WITH][0]).names))]
teams, agreement = bv.assign_teams(VIDEO, tracks[(RENDER_WITH, "bytetrack.yaml")], names.index("player"))
print(f"{len(teams)} player tracks assigned; team sizes {pd.Series(teams).value_counts().to_dict()}; "
      f"frame-vote agreement {agreement:.3f}")
raw = out / "tracked_raw.mp4"
bv.render(VIDEO, tracks[(RENDER_WITH, "bytetrack.yaml")], teams, raw, names)
final = out / "08_tracking_teams.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw), "-c:v", "libx264", "-pix_fmt", "yuv420p",
                "-crf", "23", str(final)], check=True)
raw.unlink()
(out / "tracking_stats.json").write_text(json.dumps({
    "clip": bv.DFL_CLIP_CREDIT,
    "render_tracker": "bytetrack.yaml", "fps": fps,
    "per_detector_tracker": {f"{d} | {t}": v for (d, t), v in stats.items()},
    "render_with": RENDER_WITH, "team_agreement": agreement, "player_tracks_assigned": len(teams)}, indent=2))
print(final)